# Host-level risk on LANL authentication traffic

This notebook walks through a temporal GNN on the LANL Cyber-1 auth stream: roughly speaking, which machines are about to show up in malicious-looking authentication, scored only from what we've seen up to each fixed clock tick.

I treat red-team marks as the supervised signal. same imperfect proxy everyone uses on this dataset.

> At decision time `t`, for each active host `i`: will that host appear as **either** source **or** destination on **any** malicious auth in `(t, t+H]`?

That's forecasting into the future window, not anomaly scoring on the past. I'll keep harping on **temporal leakage** because that's where notebooks usually cheat by accident.


## 1. Imports and environment setup

Plain NumPy/Pandas/Matplotlib plus PyTorch for the model. I skipped PyTorch Geometric on purpose — the event replay + per-host queues are easier to read in plain `torch`, and the Implementation Notes section says more about that trade-off.


In [ ]:
import os
import math
import json
import time
import random
import warnings
from collections import defaultdict, deque
from dataclasses import dataclass, field, asdict
from typing import Optional, Tuple, List, Dict, Iterable

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


def set_seed(seed: int) -> None:
    # Make Python, NumPy, and PyTorch deterministic enough for a tutorial.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 2. What we're modeling

Auth logs aren't a static graph; they're a **stream**. Conceptually:

```
raw log  →  chronological events e₁ … eₙ  →  memory bumps every event
           →  at fixed decision times t: snapshot hosts, attention over recent edges → risk
```

**Two clocks:**

1. **Event time** — moves whenever a row arrives; memory updates live here.
2. **Decision time** — coarse grid (every few minutes) where we actually score hosts. Nobody refreshes a SOC dashboard on every single Kerberos line.

That split matches how I'd deploy something like this: continuous ingestion, periodic scoring.

Each event is roughly \( e_k = (u_k, v_k, t_k, x_k) \): endpoints, timestamp, and engineered features from metadata plus rolling stats.


## Config — paths, windows, demo defaults

Everything tunable sits in `Config` below. If your CSV columns or paths differ from stock LANL layout, change them **here** — downstream code pulls from this object.

### Fast notebook run vs. full-scale experiment

Defaults here are meant for something that **finishes on one GPU** while you're iterating: fewer rows, shorter model dims, chunk sizing for replay, etc. Inline comments next to fields keep the heavier "paper run" numbers visible.

Rough idea of what's shrunk for speed:

- fewer raw rows streamed from `auth.txt.gz`
- wider spacing between decision times → fewer scoring steps
- smaller `K`, smaller embedding widths
- `replay_encode_chunk_size` so replay doesn't allocate one monster batch

Timing knobs (`history_window_sec`, `forecast_horizon_sec`) stay aligned with how I describe the task later.

### Caches under `artifacts_dir`

Numeric features and the `(hosts, labels)` dict per decision time are expensive to rebuild. First run writes them; later runs load unless you flip:

- `force_recompute_features`
- `force_recompute_decision_cache`


In [ ]:
@dataclass
class Config:
    # ---- File paths (USER CONFIG) -----------------------------------------
    data_dir: str = "./data"
    auth_path: str = "../project/data/auth.txt.gz"
    redteam_path: str = "../project/data/redteam.txt.gz"
    figures_dir: str = "./figures"
    artifacts_dir: str = "./artifacts"

    # ---- Column names in the raw auth file (USER CONFIG) ------------------
    # If your CSV uses different headers, edit them here.
    col_time: str = "time"
    col_src_user: str = "src_user"
    col_dst_user: str = "dst_user"
    col_src_host: str = "src_computer"
    col_dst_host: str = "dst_computer"
    col_auth_type: str = "auth_type"
    col_logon_type: str = "logon_type"
    col_auth_orient: str = "auth_orient"
    col_success: str = "success"

    # ---- Column names in the red-team file (USER CONFIG) ------------------
    rt_col_time: str = "time"
    rt_col_user: str = "user"
    rt_col_src_host: str = "src_computer"
    rt_col_dst_host: str = "dst_computer"

    # ---- Loading / subsetting --------------------------------------------
    # The full auth.txt.gz has ~1.05B rows. For one-notebook tractability we
    # load a TIME WINDOW of events anchored on the redteam time range:
    #   - load redteam first
    #   - take t_lo = redteam.time.min() - history_window_sec  (small buffer)
    #   - take t_hi = t_lo + focus_span_sec                    (e.g. 3 days)
    # then stream the auth file in chunks of `load_chunksize` rows and keep
    # only events with t in [t_lo, t_hi], capped at `sample_nrows`.
    #
    # NOTE: sample_nrows is intentionally small for a fast notebook demo.
    # The research-spec runs use 2_000_000 or more rows.
    sample_nrows: Optional[int] = 300_000        # debug demo cap (spec: 2_000_000+)
    focus_span_sec: int = 3 * 86400              # 3 days of auth around redteam
    load_chunksize: int = 1_000_000              # pandas read_csv chunksize

    # ---- Forecasting time windows (seconds) -------------------------------
    # Research spec: W = 6h, H = 30min, decision_interval = 5min.
    # We use a 1h history window and a 15-minute decision interval here so the
    # debug-sized sample yields enough decision times to train on.
    history_window_sec: int = 10 * 60           # W (demo); spec: 6 * 3600
    forecast_horizon_sec: int = 5 * 60          # H = 30 minutes (spec)
    decision_interval_sec: int = 5 * 60         # 15 minutes for fast demo (spec: 5 * 60)

    # ---- Model architecture ----------------------------------------------
    # Demo defaults are smaller and faster. Spec values are 128 across the
    # board. Increase these once the data sample is large enough to justify
    # the extra capacity.
    # K_recent_events: int = 10                    # demo; spec: 20
    # memory_dim: int = 64                         # demo; spec: 128
    cat_emb_dim: int = 16
    # host_emb_dim: int = 64                       # demo; spec: 128
    # risk_hidden_dim: int = 64
    # time_enc_dim: int = 32
    event_repr_dim: int = 64                     # demo; spec: 128
    attn_hidden_dim: int = 64                    # demo; spec: 128
    dropout: float = 0.1

    K_recent_events = 5
    memory_dim = 32
    host_emb_dim = 32
    event_repr_dim = 32
    attn_hidden_dim = 32
    risk_hidden_dim = 32
    time_enc_dim = 16

    # ---- Optimization ----------------------------------------------------
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    num_epochs: int = 10
    early_stop_patience: int = 2
    # How many decision-time prediction batches we accumulate before each
    # gradient step. Truncates BPTT through the memory chain; smaller values
    # save memory but make training noisier.
    decisions_per_grad_step: int = 4

    # ---- Data splits ------------------------------------------------------
    # We split *chronologically* by fractions of the time range present in
    # the loaded sample. A random shuffle would be a temporal-leakage bug.
    train_frac: float = 0.6
    val_frac: float = 0.2
    test_frac: float = 0.2  # implicitly 1 - train - val

    # ---- Speed / memory knobs --------------------------------------------
    # We move event tensors to GPU ONCE and reuse the cached device tensors
    # everywhere instead of repeatedly calling `.to(DEVICE)` inside hot loops.
    # During event replay we encode events in CHUNKS of this size rather than
    # one giant batch covering the whole (prev_dt, dt] window: the per-event
    # GRU memory updates are still applied sequentially (events ordering
    # matters) but the encoder pass is bounded in GPU memory, which is what
    # makes scaling beyond a small sample tractable.
    # replay_encode_chunk_size: int = 4096
    replay_encode_chunk_size: int = 2048

    # ---- Caching flags ---------------------------------------------------
    # Numeric per-event features and the precomputed decision-example cache
    # are saved to disk under artifacts_dir. On rerun we load from disk
    # instead of recomputing (which dominates wall time on warm cache).
    # Set these flags to True to override and recompute.
    force_recompute_features: bool = False
    force_recompute_decision_cache: bool = False

    # ---- Misc ------------------------------------------------------------
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    verbose: bool = True

    # ---- Internal: how many decision times to subsample for sanity pass --
    sanity_max_decisions: int = 8


CFG = Config()
os.makedirs(CFG.figures_dir, exist_ok=True)
os.makedirs(CFG.artifacts_dir, exist_ok=True)
set_seed(CFG.seed)
DEVICE = torch.device(CFG.device)

print("Config:")
print(json.dumps(asdict(CFG), indent=2, default=str))
print("Device:", DEVICE)

## 3. Loading LANL auth + red-team

`auth.txt.gz` is headerless CSV; usual column order:

| idx | field |
|-----|-------|
| 0 | time (s) |
| 1–2 | src/dst user |
| 3–4 | src/dst computer |
| 5–8 | auth metadata + success |

`redteam.txt.gz` matches the usual LANL competition layout (`time`, `user`, src/dst host). Missing values show up as `?`.

Loaders match what I used in `lanl_cyber1_eda.ipynb` — same `read_csv` contract. There's also `load_auth_focus_window`, which walks the gzip in chunks and keeps only timestamps inside a chosen interval.

**Why not just `nrows=2e6` from the top of the file?** The dump is ~58 days long; the first couple million **rows** only cover a few hours of wall time, while red-team activity starts around day ~2. A naive head sample often has **zero** positives. Anchoring a time window on red-team timestamps keeps malicious rows in frame.


In [ ]:
# NOTE: This block is a direct port of the loading pattern in
# `lanl_cyber1_eda.ipynb` (same `pd.read_csv` arguments, same column names,
# same `?` -> NaN handling). The Config column names default to exactly these,
# so the rest of the notebook (which references `CFG.col_*`) keeps working.
#
# We additionally provide `load_auth_focus_window`, which uses chunked
# iteration to slice the auth file by *time* (in seconds since the LANL
# epoch). This is important because the first ~2M rows of auth.txt.gz only
# cover ~5.7 hours, while red-team activity starts at ~1.75 days. A naive
# `nrows=2_000_000` head sample therefore contains ZERO malicious events
# and the supervised model has nothing to learn.

AUTH_COLS = [
    "time", "src_user", "dst_user", "src_computer", "dst_computer",
    "auth_type", "logon_type", "auth_orient", "success",
]
REDTEAM_COLS = ["time", "user", "src_computer", "dst_computer"]


def load_auth_sample(path, nrows=None, chunksize=None, **kwargs):
    # Read auth.txt.gz: `nrows` for a head sample; `chunksize` for a chunked
    # iterator. Mirrors `load_auth_sample` in the EDA notebook.
    kw = dict(
        compression="gzip",
        header=None,
        names=AUTH_COLS,
        na_values=["?"],
        low_memory=False,
    )
    kw.update(kwargs)
    if chunksize:
        return pd.read_csv(path, chunksize=chunksize, **kw)
    return pd.read_csv(path, nrows=nrows, **kw)


def load_redteam(path):
    # Read redteam.txt.gz with the same conventions as the EDA notebook.
    return pd.read_csv(
        path,
        compression="gzip",
        header=None,
        names=REDTEAM_COLS,
        na_values=["?"],
        low_memory=False,
    )


def load_auth_focus_window(
    path: str,
    t_lo: int,
    t_hi: int,
    max_rows: Optional[int] = None,
    chunksize: int = 1_000_000,
    verbose: bool = True,
) -> pd.DataFrame:
    # Stream auth.txt.gz in chunks and keep only events with t_lo <= time <= t_hi.
    # Stops early once we've collected `max_rows` rows or once we move past t_hi
    # (auth.txt.gz is already sorted by time, so once chunk.time.min() > t_hi we
    # can stop reading). This is the recommended loader for the modeling
    # pipeline because it guarantees the sample overlaps red-team activity.
    pieces: List[pd.DataFrame] = []
    kept = 0
    reader = load_auth_sample(path, chunksize=chunksize)
    for i, chunk in enumerate(reader):
        # `time` is integer seconds; convert defensively.
        t = chunk["time"].astype(np.int64)
        if t.iloc[0] > t_hi:
            if verbose:
                print(f"  chunk {i}: starts at t={int(t.iloc[0])} > t_hi={t_hi}, stopping")
            break
        if t.iloc[-1] < t_lo:
            if verbose and (i % 10 == 0):
                print(f"  chunk {i}: ends at t={int(t.iloc[-1])} < t_lo={t_lo}, skipping")
            continue
        sub = chunk[(t >= t_lo) & (t <= t_hi)]
        if len(sub):
            pieces.append(sub)
            kept += len(sub)
            if verbose:
                print(f"  chunk {i}: kept {len(sub):,} rows (running total={kept:,})")
        if max_rows is not None and kept >= max_rows:
            if verbose:
                print(f"  reached max_rows={max_rows:,}, stopping")
            break
    if not pieces:
        raise RuntimeError(
            f"No auth rows found in [t_lo={t_lo}, t_hi={t_hi}]. Check the time range."
        )
    df = pd.concat(pieces, ignore_index=True)
    if max_rows is not None and len(df) > max_rows:
        df = df.iloc[:max_rows].copy()
    return df


def expect_columns(df: pd.DataFrame, expected: List[str], name: str) -> None:
    # Diagnostic helper: raise with a helpful message if columns disagree.
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(
            f"{name} is missing expected columns: {missing}. "
            f"Got: {list(df.columns)}."
        )
    print(f"[{name}] columns OK: {expected}")

Next cells pull auth + redteam, print shapes, peek at rows, and sanity-check categoricals — same drill as any first-pass EDA.


In [ ]:
# --- 1. Load redteam (small file, used to anchor the auth time window) ---
print("Loading redteam...")
redteam_raw = load_redteam(CFG.redteam_path)
print("redteam shape:", redteam_raw.shape)
print(redteam_raw.head())
print()

rt_t_min = int(redteam_raw[CFG.rt_col_time].min())
rt_t_max = int(redteam_raw[CFG.rt_col_time].max())
print(f"redteam time range (s): {rt_t_min} -> {rt_t_max} "
      f"({rt_t_min / 86400:.2f}d -> {rt_t_max / 86400:.2f}d since epoch)")

# --- 2. Load auth events around the redteam time range ------------------
# The first ~2M rows of auth.txt.gz only cover ~5.7 hours, but redteam
# activity begins at ~1.74 days. A pure head sample would have 0 positive
# labels. We instead pull a chronological slice of auth that *contains*
# redteam activity, capped at CFG.sample_nrows.
T_LO = max(0, rt_t_min - CFG.history_window_sec)
T_HI = T_LO + CFG.focus_span_sec
print()
print(f"Loading auth events in window [t_lo={T_LO}, t_hi={T_HI}] "
      f"(span {(T_HI - T_LO) / 86400:.2f} days, cap={CFG.sample_nrows:,} rows)...")
auth_raw = load_auth_focus_window(
    CFG.auth_path,
    t_lo=T_LO,
    t_hi=T_HI,
    max_rows=CFG.sample_nrows,
    chunksize=CFG.load_chunksize,
    verbose=True,
)
print()
print("auth shape:", auth_raw.shape)
print(auth_raw.head())
print()

# Sanity diagnostics: downstream cells use CFG.col_* names. With the Config
# defaults those equal AUTH_COLS / REDTEAM_COLS exactly.
expected_auth = [
    CFG.col_time, CFG.col_src_user, CFG.col_dst_user,
    CFG.col_src_host, CFG.col_dst_host,
    CFG.col_auth_type, CFG.col_logon_type, CFG.col_auth_orient, CFG.col_success,
]
expected_rt = [CFG.rt_col_time, CFG.rt_col_user, CFG.rt_col_src_host, CFG.rt_col_dst_host]
expect_columns(auth_raw, expected_auth, "auth")
expect_columns(redteam_raw, expected_rt, "redteam")

# How many redteam events fall inside the auth window we loaded?
rt_in_window = redteam_raw[
    (redteam_raw[CFG.rt_col_time] >= T_LO) & (redteam_raw[CFG.rt_col_time] <= T_HI)
]
print()
print(f"redteam events inside loaded window: {len(rt_in_window)}/{len(redteam_raw)}")
print(f"auth time range loaded (s): {int(auth_raw[CFG.col_time].min())} -> "
      f"{int(auth_raw[CFG.col_time].max())}")
print()
print("Per-column NaN counts (auth):")
print(auth_raw.isna().sum())
print()
print("Unique value counts (small categoricals):")
for c in [CFG.col_auth_type, CFG.col_logon_type, CFG.col_auth_orient, CFG.col_success]:
    if c in auth_raw.columns:
        print(f"  {c}: {auth_raw[c].nunique()} unique")

## 4. Cleaning

Rough pipeline:

1. Drop rows missing time or either host — nothing to attach to the graph without them.
2. Stable sort by time (everything downstream assumes chronological order).
3. Split `user@domain` when `@` is present; otherwise leave user string alone and leave domain empty.
4. Point missing categoricals at `"<UNK>"` so they get an embedding slot.
5. Tag malicious rows by joining auth to red-team on `(time, src_user, src_host, dst_host)` — same join pattern as the usual LANL benchmark recipes.

Strings stay as strings for one more section; integer IDs come next.


In [ ]:
def clean_auth(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    # Drop rows with missing time / src_host / dst_host
    before = len(df)
    df = df.dropna(subset=[cfg.col_time, cfg.col_src_host, cfg.col_dst_host]).copy()
    df[cfg.col_time] = df[cfg.col_time].astype(np.int64)
    df = df.sort_values(cfg.col_time, kind="mergesort").reset_index(drop=True)
    after = len(df)
    print(f"clean_auth: kept {after}/{before} rows after dropping missing critical fields")

    # Split src_user into (src_user_only, src_domain) when '@' is present
    def split_user_domain(series: pd.Series) -> Tuple[pd.Series, pd.Series]:
        s = series.astype(str).fillna("<UNK>")
        parts = s.str.split("@", n=1, expand=True)
        user = parts[0].fillna("<UNK>")
        domain = parts[1] if parts.shape[1] > 1 else pd.Series([""] * len(s))
        domain = domain.fillna("")
        return user, domain

    src_u, src_d = split_user_domain(df[cfg.col_src_user])
    dst_u, dst_d = split_user_domain(df[cfg.col_dst_user])
    df["src_user_only"] = src_u
    df["src_domain"] = src_d
    df["dst_user_only"] = dst_u
    df["dst_domain"] = dst_d

    # Fill remaining missing categoricals with explicit "<UNK>"
    cat_cols = [
        cfg.col_auth_type, cfg.col_logon_type, cfg.col_auth_orient, cfg.col_success,
        "src_user_only", "src_domain",
    ]
    for c in cat_cols:
        df[c] = df[c].astype(str).fillna("<UNK>")
        df.loc[df[c] == "nan", c] = "<UNK>"

    return df


def attach_redteam_label(auth_df: pd.DataFrame, rt_df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    # Mark rows that match a red-team event on (time, src_user, src_host, dst_host).
    rt = rt_df.dropna().copy()
    rt[cfg.rt_col_time] = rt[cfg.rt_col_time].astype(np.int64)
    rt = rt.rename(columns={cfg.rt_col_user: cfg.col_src_user})
    keys = [cfg.col_time, cfg.col_src_user, cfg.col_src_host, cfg.col_dst_host]
    merged = auth_df.merge(
        rt[keys].drop_duplicates(),
        on=keys,
        how="left",
        indicator=True,
    )
    auth_df = auth_df.copy()
    auth_df["is_malicious"] = (merged["_merge"] == "both").to_numpy()
    return auth_df


auth_clean = clean_auth(auth_raw, CFG)
auth_clean = attach_redteam_label(auth_clean, redteam_raw, CFG)
print()
print("auth_clean shape:", auth_clean.shape)
print("malicious rows in sample:", int(auth_clean["is_malicious"].sum()))
print()
print("auth_clean head:")
print(auth_clean.head(3))

## 5. Event stream

From the cleaned frame I keep one row per auth line, sorted by time, carrying host ids once they're assigned and the malicious flag for labels later.

There’s a quick plot of event density (and malicious density) so split boundaries aren’t chosen blind.


In [ ]:
def make_chronological_event_table(auth_df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    df = auth_df[[
        cfg.col_time,
        cfg.col_src_host, cfg.col_dst_host,
        cfg.col_auth_type, cfg.col_logon_type, cfg.col_auth_orient, cfg.col_success,
        "src_user_only", "src_domain",
        "is_malicious",
    ]].copy()
    df = df.sort_values(cfg.col_time, kind="mergesort").reset_index(drop=True)
    return df


events_df = make_chronological_event_table(auth_clean, CFG)
print("events_df shape:", events_df.shape)
print("time range (s):", int(events_df[CFG.col_time].min()), "to", int(events_df[CFG.col_time].max()))
print()
print(events_df.head())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(events_df[CFG.col_time].values // 3600, bins=80, color="steelblue", alpha=0.85)
axes[0].set_xlabel("Hour index"); axes[0].set_ylabel("Events"); axes[0].set_title("Auth event volume by hour (sample)")
mal_t = events_df.loc[events_df["is_malicious"], CFG.col_time].values
if len(mal_t) > 0:
    axes[1].hist(mal_t // 3600, bins=80, color="crimson", alpha=0.85)
axes[1].set_xlabel("Hour index"); axes[1].set_ylabel("Malicious events"); axes[1].set_title("Malicious event volume by hour")
plt.tight_layout()
plt.savefig(os.path.join(CFG.figures_dir, "event_volume.png"), dpi=120)
plt.show()

## 6. Per-event features

Each event gets a vector \(x_k\) in two buckets:

**Categoricals** (embedded inside the model): `auth_type`, `logon_type`, `auth_orient`, `success`, split-out user + domain fields. I integer-encode now; the module learns tables instead of giant one-hots.

**Numerics** (causal — only past events):

- gaps since this host last appeared as src / dst
- rolling success/fail counts and rough degree proxies in a trailing window that lines up with history window \(W\)

Features are computed in one forward scan with per-host pointers so nothing peeks forward. Count features are heavy-tailed, so `log1p` + train-only standardization show up a few cells later.


In [ ]:
# ---- 6.1 Integer encode hosts and categorical event fields ----------------

def fit_int_encoder(values: Iterable[str], reserve_unk: bool = True) -> Dict[str, int]:
    """Map each unique value to a stable integer id.

    If `reserve_unk`, id 0 is reserved for `<UNK>` so that unseen values at
    scoring time map to a known slot. We ignore NaNs in the input.
    """
    uniq = sorted({str(v) for v in values if pd.notna(v)})
    enc: Dict[str, int] = {}
    next_id = 0
    if reserve_unk:
        enc["<UNK>"] = 0
        next_id = 1
    for v in uniq:
        if reserve_unk and v == "<UNK>":
            continue
        enc[v] = next_id
        next_id += 1
    return enc


def apply_int_encoder(values: Iterable[str], enc: Dict[str, int]) -> np.ndarray:
    # Map values through the encoder, defaulting to 0 (UNK) for unseen.
    return np.fromiter((enc.get(str(v), 0) for v in values), dtype=np.int64, count=len(values))


# Build host vocab from BOTH src and dst columns so a host has the same id either way.
all_hosts = pd.unique(pd.concat([
    events_df[CFG.col_src_host].astype(str),
    events_df[CFG.col_dst_host].astype(str),
], ignore_index=True))
host2id: Dict[str, int] = {h: i for i, h in enumerate(sorted(all_hosts))}
id2host: Dict[int, str] = {i: h for h, i in host2id.items()}
N_HOSTS = len(host2id)

# Categorical fields and their encoders (fit on the loaded sample only;
# UNK is reserved for runtime-unseen values).
CAT_FIELDS = [
    CFG.col_auth_type,
    CFG.col_logon_type,
    CFG.col_auth_orient,
    CFG.col_success,
    "src_user_only",
    "src_domain",
]
cat_encoders: Dict[str, Dict[str, int]] = {
    f: fit_int_encoder(events_df[f].astype(str).unique()) for f in CAT_FIELDS
}
cat_vocab_sizes: Dict[str, int] = {f: max(cat_encoders[f].values()) + 1 for f in CAT_FIELDS}

print("N_HOSTS:", N_HOSTS)
print("Categorical vocab sizes:", cat_vocab_sizes)

In [ ]:
# ---- 6.2 Numeric features: time gaps + rolling W-second counts ------------
# We compute features in a SINGLE chronological pass to guarantee no leakage.
# For the rolling W-second windows we maintain per-host deques of timestamps
# and pop from the left whenever the head is older than W.
#
# Earlier this cell had a slow duplicate implementation; now it's one path with
# parquet cache under `CFG.artifacts_dir`. Set `CFG.force_recompute_features`
# to ignore the cache and recompute.

from collections import defaultdict, deque
from tqdm import tqdm

W_SEC = CFG.history_window_sec


def compute_event_features_fast(events_df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """One-pass per-event feature computation.

    For every event k we record:
      - time gaps since the host's previous appearance (src and dst),
      - rolling W-second counts of successful and failed events as source,
      - rolling W-second unique-counterpart out-degree (src) and in-degree (dst).

    We use per-host deques + per-counterpart count maps so the rolling unique
    counterpart count is O(1) amortized rather than O(K) per event.
    """
    n = len(events_df)
    t_arr_local = events_df[cfg.col_time].to_numpy(dtype=np.int64)
    src_arr_local = apply_int_encoder(events_df[cfg.col_src_host].astype(str), host2id)
    dst_arr_local = apply_int_encoder(events_df[cfg.col_dst_host].astype(str), host2id)
    succ_str = events_df[cfg.col_success].astype(str).str.lower().to_numpy()
    is_success_event = succ_str == "success"

    last_seen = np.full(N_HOSTS, -1, dtype=np.int64)
    src_succ_buf = defaultdict(deque)
    src_fail_buf = defaultdict(deque)
    src_outdeg_buf = defaultdict(deque)   # (time, counterpart)
    dst_indeg_buf = defaultdict(deque)    # (time, counterpart)
    src_outdeg_counts = defaultdict(lambda: defaultdict(int))
    dst_indeg_counts = defaultdict(lambda: defaultdict(int))
    src_outdeg_unique = defaultdict(int)
    dst_indeg_unique = defaultdict(int)

    dt_src_prev = np.zeros(n, dtype=np.float32)
    dt_dst_prev = np.zeros(n, dtype=np.float32)
    roll_src_succ = np.zeros(n, dtype=np.float32)
    roll_src_fail = np.zeros(n, dtype=np.float32)
    roll_src_outdeg = np.zeros(n, dtype=np.float32)
    roll_dst_indeg = np.zeros(n, dtype=np.float32)

    SENTINEL_GAP = float(W_SEC * 4)

    for k in tqdm(range(n), desc="compute_event_features_fast"):
        t = int(t_arr_local[k])
        s = int(src_arr_local[k])
        d = int(dst_arr_local[k])

        dt_src_prev[k] = SENTINEL_GAP if last_seen[s] < 0 else (t - last_seen[s])
        dt_dst_prev[k] = SENTINEL_GAP if last_seen[d] < 0 else (t - last_seen[d])

        cutoff = t - W_SEC

        b = src_succ_buf[s]
        while b and b[0] < cutoff:
            b.popleft()
        b = src_fail_buf[s]
        while b and b[0] < cutoff:
            b.popleft()

        b = src_outdeg_buf[s]
        counts = src_outdeg_counts[s]
        while b and b[0][0] < cutoff:
            _, old_dst = b.popleft()
            counts[old_dst] -= 1
            if counts[old_dst] == 0:
                del counts[old_dst]
                src_outdeg_unique[s] -= 1

        b = dst_indeg_buf[d]
        counts = dst_indeg_counts[d]
        while b and b[0][0] < cutoff:
            _, old_src = b.popleft()
            counts[old_src] -= 1
            if counts[old_src] == 0:
                del counts[old_src]
                dst_indeg_unique[d] -= 1

        roll_src_succ[k] = len(src_succ_buf[s])
        roll_src_fail[k] = len(src_fail_buf[s])
        roll_src_outdeg[k] = src_outdeg_unique[s]
        roll_dst_indeg[k] = dst_indeg_unique[d]

        if is_success_event[k]:
            src_succ_buf[s].append(t)
        else:
            src_fail_buf[s].append(t)

        src_outdeg_buf[s].append((t, d))
        if src_outdeg_counts[s][d] == 0:
            src_outdeg_unique[s] += 1
        src_outdeg_counts[s][d] += 1

        dst_indeg_buf[d].append((t, s))
        if dst_indeg_counts[d][s] == 0:
            dst_indeg_unique[d] += 1
        dst_indeg_counts[d][s] += 1

        last_seen[s] = t
        last_seen[d] = t

    return pd.DataFrame({
        "dt_src_prev": np.log1p(dt_src_prev),
        "dt_dst_prev": np.log1p(dt_dst_prev),
        "roll_src_succ_6h": np.log1p(roll_src_succ),
        "roll_src_fail_6h": np.log1p(roll_src_fail),
        "roll_src_outdeg_6h": np.log1p(roll_src_outdeg),
        "roll_dst_indeg_6h": np.log1p(roll_dst_indeg),
    })


def _numeric_features_cache_path(cfg: Config, n_events: int) -> str:
    # Filename includes the parameters that affect the computation so that a
    # change in W or the loaded sample size invalidates the cache automatically.
    fname = f"numeric_features__nrows{n_events}_W{cfg.history_window_sec}.parquet"
    return os.path.join(cfg.artifacts_dir, fname)


def load_or_compute_numeric_features(events_df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Load cached per-event features from parquet if available, else compute and save.

    The cache key embeds n_events and W in the filename so it auto-invalidates
    when the loaded sample or rolling-window length changes.
    """
    cache_path = _numeric_features_cache_path(cfg, len(events_df))
    if (not cfg.force_recompute_features) and os.path.exists(cache_path):
        feats = pd.read_parquet(cache_path)
        if len(feats) == len(events_df):
            print(f"[numeric_features] loaded from cache: {cache_path}")
            return feats
        print(f"[numeric_features] cache size mismatch, recomputing: {cache_path}")
    print(f"[numeric_features] computing fresh ({len(events_df):,} events)...")
    feats = compute_event_features_fast(events_df, cfg)
    feats.to_parquet(cache_path)
    print(f"[numeric_features] saved to: {cache_path}")
    return feats


numeric_features = load_or_compute_numeric_features(events_df, CFG)
print("numeric_features shape:", numeric_features.shape)
print(numeric_features.describe())

In [ ]:
# ---- 6.3 Encode categoricals into integer arrays --------------------------
cat_int = {
    f: apply_int_encoder(events_df[f].astype(str), cat_encoders[f])
    for f in CAT_FIELDS
}
src_id_arr = apply_int_encoder(events_df[CFG.col_src_host].astype(str), host2id)
dst_id_arr = apply_int_encoder(events_df[CFG.col_dst_host].astype(str), host2id)
t_arr = events_df[CFG.col_time].to_numpy(dtype=np.int64)
malicious_arr = events_df["is_malicious"].to_numpy(dtype=bool)

print("src_id_arr dtype/shape:", src_id_arr.dtype, src_id_arr.shape)
print("Per-categorical example head:")
for f in CAT_FIELDS:
    print(f, cat_int[f][:5])

### Quick feature plots

Histograms for gaps and rolling counts — mostly to catch silent bugs and insane scales before training eats them.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, numeric_features.columns):
    ax.hist(numeric_features[col].values, bins=60, color="steelblue", alpha=0.85)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(os.path.join(CFG.figures_dir, "numeric_feature_hists.png"), dpi=120)
plt.show()

## 7. Decision times and labels

I split the timeline into decision ticks every `decision_interval_sec`. At time \(t\):

- anything with timestamp \(\le t\) is fair game for the model;
- I only score hosts that fired at least once in \((t-W, t]\);
- label for host \(i\) is 1 iff it touches a malicious row in \((t, t+H]\).

That's an operational proxy ("about to co-occur with bad traffic"), not a perfect "compromised" label — worth remembering when metrics look weird.

Implementation-wise I build a feasible decision grid, use binary search on sorted host timelines for "active", and cache malicious hits per host so labels stay cheap. That avoids materializing every `(host, decision)` pair explicitly.


In [ ]:
# ---- 7.1 Decision-time grid ----------------------------------------------
T_MIN = int(events_df[CFG.col_time].min())
T_MAX = int(events_df[CFG.col_time].max())
DEC_STEP = CFG.decision_interval_sec
H_SEC = CFG.forecast_horizon_sec

dec_start = T_MIN + W_SEC
dec_end = T_MAX - H_SEC
decision_times = np.arange(dec_start, dec_end + 1, DEC_STEP, dtype=np.int64)
print(f"Decision times: {len(decision_times)} (every {DEC_STEP}s, "
      f"from {dec_start} to {dec_end})")

if len(decision_times) < 3:
    raise RuntimeError(
        "Loaded sample is too short for the configured (W, H, decision_interval). "
        f"Span = {T_MAX - T_MIN}s, W = {W_SEC}s, H = {H_SEC}s. "
        "Either increase Config.sample_nrows or shrink Config.history_window_sec."
    )

In [ ]:
# ---- 7.2 Per-host index of involvement times -----------------------------
# For each host, store the sorted timestamps when it appeared (as src OR dst).
host_appearance_times: List[np.ndarray] = [None] * N_HOSTS
host_malicious_times: List[np.ndarray] = [None] * N_HOSTS

# Build via a single pass.
appear_lists = [[] for _ in range(N_HOSTS)]
mal_lists = [[] for _ in range(N_HOSTS)]
for k in range(len(t_arr)):
    s = int(src_id_arr[k]); d = int(dst_id_arr[k]); t = int(t_arr[k])
    appear_lists[s].append(t)
    if d != s:
        appear_lists[d].append(t)
    if malicious_arr[k]:
        mal_lists[s].append(t)
        if d != s:
            mal_lists[d].append(t)
for h in range(N_HOSTS):
    host_appearance_times[h] = np.array(sorted(appear_lists[h]), dtype=np.int64)
    host_malicious_times[h] = np.array(sorted(mal_lists[h]), dtype=np.int64)

n_active_hosts_with_mal = sum(1 for a in host_malicious_times if len(a) > 0)
print("Hosts ever active in sample:",
      sum(1 for a in host_appearance_times if len(a) > 0))
print("Hosts ever in a malicious event:", n_active_hosts_with_mal)

In [ ]:
# ---- 7.3 Helpers: active hosts in a window, and host label for (h, t) -----
#
# `active_hosts_in_window` is fine for preprocessing, but calling it inside the
# training loop used to dominate runtime. We only use it from
# `precompute_decision_examples` below.

def active_hosts_in_window(t: int, w: int) -> np.ndarray:
    """Hosts with at least one appearance in (t - w, t]."""
    ids = []
    lo = t - w
    for h in range(N_HOSTS):
        arr = host_appearance_times[h]
        if arr.size == 0:
            continue
        i_lo = np.searchsorted(arr, lo, side="right")
        i_hi = np.searchsorted(arr, t, side="right")
        if i_hi > i_lo:
            ids.append(h)
    return np.array(ids, dtype=np.int64)


def host_label_at(t: int, h: int, horizon: int) -> int:
    """1 if host h appears in a malicious event in (t, t + horizon], else 0."""
    arr = host_malicious_times[h]
    if arr.size == 0:
        return 0
    i_lo = np.searchsorted(arr, t, side="right")          # strictly after t
    i_hi = np.searchsorted(arr, t + horizon, side="right")
    return int(i_hi > i_lo)


def precompute_decision_examples(
    decision_times_arr: np.ndarray,
    cfg: Config,
) -> Dict[int, Tuple[np.ndarray, np.ndarray]]:
    """Precompute (active hosts, labels) for every decision time, ONCE.

    Returns a dict mapping decision time (int seconds) -> (hosts, labels)
    where `hosts` is a NumPy int64 array of host ids and `labels` is a
    NumPy int8 array of 0/1 values.

    This replaces the old per-epoch pattern of calling
    `active_hosts_in_window` + `host_label_at` inside the training loop.
    The training and evaluation code reads from the returned cache directly,
    so the per-host scanning cost is paid exactly once per process (and
    optionally amortized across runs via `_decision_cache_path`).
    """
    out: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}
    for dt in tqdm(decision_times_arr, desc="precompute_decision_examples"):
        dt_i = int(dt)
        hosts = active_hosts_in_window(dt_i, cfg.history_window_sec)
        if hosts.size == 0:
            continue
        labels = np.fromiter(
            (host_label_at(dt_i, int(h), cfg.forecast_horizon_sec) for h in hosts),
            dtype=np.int8, count=len(hosts),
        )
        out[dt_i] = (hosts, labels)
    return out


def _decision_cache_path(cfg: Config, n_events: int, n_decisions: int) -> str:
    # Cache key embeds the parameters that affect the computation. If any of
    # these change, a new cache file is used and the old one is ignored.
    fname = (
        f"decision_examples__nrows{n_events}"
        f"_W{cfg.history_window_sec}"
        f"_H{cfg.forecast_horizon_sec}"
        f"_step{cfg.decision_interval_sec}"
        f"_nd{n_decisions}.pkl"
    )
    return os.path.join(cfg.artifacts_dir, fname)


def load_or_compute_decision_cache(
    decision_times_arr: np.ndarray,
    cfg: Config,
    n_events: int,
) -> Dict[int, Tuple[np.ndarray, np.ndarray]]:
    """Disk-cached wrapper around `precompute_decision_examples`."""
    import pickle
    cache_path = _decision_cache_path(cfg, n_events, len(decision_times_arr))
    if (not cfg.force_recompute_decision_cache) and os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)
        print(f"[decision_examples] loaded from cache: {cache_path} "
              f"({len(cache)} non-empty decision times)")
        return cache
    print(f"[decision_examples] computing fresh ({len(decision_times_arr)} dts)...")
    cache = precompute_decision_examples(decision_times_arr, cfg)
    with open(cache_path, "wb") as f:
        pickle.dump(cache, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"[decision_examples] saved to: {cache_path}")
    return cache


print("Helpers ready.")

## 8. Tensors + host state

Three pieces:

1. **Event tensors** — `event_t`, src/dst ids, categorical indices, numeric matrix (train-normalized), malicious flag.
2. **`mem_table`** — one row per host; GRU-style updates as events stream by. Gradients through replay are intentionally limited (see training notes / Implementation Notes).
3. **`host_history`** — deque of last \(K\) interactions per host: event index, time, and a **snapshot** of the other endpoint's memory at that moment (`detach`’d so it's data, not a parameter).

Normalization uses train-only stats so val/test don't leak.


In [ ]:
# ---- 8.1 Chronological train/val/test split (NO RANDOM SHUFFLE) -----------
T_RANGE = T_MAX - T_MIN
TRAIN_END_T = int(T_MIN + CFG.train_frac * T_RANGE)
VAL_END_T = int(T_MIN + (CFG.train_frac + CFG.val_frac) * T_RANGE)

train_event_mask = t_arr <= TRAIN_END_T
val_event_mask = (t_arr > TRAIN_END_T) & (t_arr <= VAL_END_T)
test_event_mask = t_arr > VAL_END_T

train_dec_mask = (decision_times <= TRAIN_END_T)
val_dec_mask = (decision_times > TRAIN_END_T) & (decision_times <= VAL_END_T)
test_dec_mask = decision_times > VAL_END_T

print("Train events:", int(train_event_mask.sum()), "| Val:", int(val_event_mask.sum()),
      "| Test:", int(test_event_mask.sum()))
print("Train decision times:", int(train_dec_mask.sum()),
      "| Val:", int(val_dec_mask.sum()), "| Test:", int(test_dec_mask.sum()))
print("Boundaries (s):", TRAIN_END_T, VAL_END_T)

### Precomputing `(hosts, labels)` per decision time

Training used to call `active_hosts_in_window` + `host_label_at` **inside every epoch**, which meant scanning hosts over and over. That’s now done **once**: build a dict `dt → (hosts, labels)`, optionally pickle it, then slice into train/val/test masks.

After this cell the hot loop only does dict lookups — the helpers above become debugging utilities, not per-step hot path.


In [ ]:
# ---- 8.1b One-time precompute of decision-time (hosts, labels) cache ------
# This is the single biggest speedup vs. the previous notebook: instead of
# scanning every host on every decision time on every epoch, we compute the
# full {dt: (hosts, labels)} dictionary ONCE (with on-disk caching) and slice
# it by the train/val/test masks. The hot training loop then only does a
# dict lookup per decision time.

DEC_TRAIN = decision_times[train_dec_mask]
DEC_VAL = decision_times[val_dec_mask]
DEC_TEST = decision_times[test_dec_mask]

all_examples = load_or_compute_decision_cache(decision_times, CFG, n_events=len(t_arr))

train_examples: Dict[int, Tuple[np.ndarray, np.ndarray]] = {
    int(dt): all_examples[int(dt)] for dt in DEC_TRAIN if int(dt) in all_examples
}
val_examples: Dict[int, Tuple[np.ndarray, np.ndarray]] = {
    int(dt): all_examples[int(dt)] for dt in DEC_VAL if int(dt) in all_examples
}
test_examples: Dict[int, Tuple[np.ndarray, np.ndarray]] = {
    int(dt): all_examples[int(dt)] for dt in DEC_TEST if int(dt) in all_examples
}


def _split_stats(name: str, ex: Dict[int, Tuple[np.ndarray, np.ndarray]]) -> None:
    n_dts = len(ex)
    n_examples = sum(len(h) for h, _ in ex.values())
    n_pos = sum(int(l.sum()) for _, l in ex.values())
    print(f"  {name:5s}: {n_dts:5d} decision times, {n_examples:8d} (host, dt) "
          f"examples, {n_pos:6d} positives")


print(f"Decision-example cache populated. Per-split sizes:")
_split_stats("train", train_examples)
_split_stats("val", val_examples)
_split_stats("test", test_examples)

In [ ]:
# ---- 8.2 Standardize numeric features using TRAIN-ONLY statistics ---------
num_train = numeric_features.loc[train_event_mask]
num_mean = num_train.mean().to_numpy()
num_std = num_train.std().replace(0, 1.0).to_numpy()
print("Train numeric means:", num_mean)
print("Train numeric stds :", num_std)

num_norm = (numeric_features.to_numpy() - num_mean) / num_std

In [ ]:
# ---- 8.3 Move event tensors to torch ---------------------------------------
#
# Old code re-uploaded the full event matrix every step; that was most of the
# wall time. These device tensors are indexed in the hot path instead.
#
# We now create *persistent device-side copies* of the per-event categorical
# and numeric tensors ONCE, right next to the CPU tensors. The downstream
# code reads from these device tensors directly (`event_cat_dev[f][idx_dev]`
# / `event_num_dev[idx_dev]`), so each replay step does exactly one tiny
# index gather on the GPU and zero host->device transfers.
#
# We deliberately keep `event_t`, `event_src`, `event_dst`, `event_malicious`
# on CPU because they are used for lightweight Python bookkeeping
# (`event_t[idx_cpu].numpy()`, last-seen lookups, label construction) and
# never participate in model computation. Mixing CPU/GPU here is intentional:
# CPU indices for CPU tensors, GPU indices for GPU tensors.

# CPU tensors (used for indexing / Python bookkeeping)
event_t = torch.from_numpy(t_arr.astype(np.int64))                                  # [E]
event_src = torch.from_numpy(src_id_arr.astype(np.int64))                          # [E]
event_dst = torch.from_numpy(dst_id_arr.astype(np.int64))                          # [E]
event_num = torch.from_numpy(num_norm.astype(np.float32))                          # [E, F_num]
event_cat = {f: torch.from_numpy(cat_int[f].astype(np.int64)) for f in CAT_FIELDS}  # each [E]
event_malicious = torch.from_numpy(malicious_arr.astype(np.bool_))                 # [E]

# Persistent device-side copies (used for ALL model inputs in the hot loop)
# WARNING: these tensors live on the GPU for the entire run. With the demo
# defaults their footprint is small (a few hundred MB). For very large
# samples you may want to keep them on CPU and accept the per-call transfer.
event_num_dev = event_num.to(DEVICE)
event_cat_dev = {f: event_cat[f].to(DEVICE) for f in CAT_FIELDS}

print("event_t shape:", event_t.shape, "dtype:", event_t.dtype)
print("event_src shape:", event_src.shape)
print("event_num shape:", event_num.shape, "device:", event_num.device,
      "| event_num_dev device:", event_num_dev.device)
for f in CAT_FIELDS:
    print(f"event_cat[{f!r}] shape:", event_cat[f].shape,
          "| dev:", event_cat_dev[f].device)
print("event_malicious shape:", event_malicious.shape)

## 9. Model pieces

Broken into small `nn.Module`s:

1. `TimeEncoding` — sine/cos gap encoding (continuous-time flavor).
2. `EventFeatureEncoder` — cats + nums → one vector per event.
3. `HostMemoryModule` — message + `GRUCell` bump per edge endpoint.
4. `TemporalAttention` — query = current memory; keys/vals from last \(K\) edges.
5. `RiskHead` — tiny MLP to a logit.
6. `TemporalHostRiskModel` — wires it together.

**Why `GRUCell` instead of batched `GRU`?** Events aren’t equally spaced; I feed gaps explicitly. Batching fixed-length sequences would fight how the stream actually arrives.


In [ ]:
class TimeEncoding(nn.Module):
    # TGN-style learnable sinusoidal time encoding.
    # Input:  dt of shape [...]   (seconds since some reference)
    # Output: [..., dim]
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        # Log-spaced initial frequencies covering ~1s to ~1e9s
        freqs = 1.0 / (10.0 ** np.linspace(0, 9, dim))
        self.basis_freq = nn.Parameter(torch.from_numpy(freqs).float())
        self.phase = nn.Parameter(torch.zeros(dim))

    def forward(self, dt: torch.Tensor) -> torch.Tensor:
        # dt can be any shape; we add a trailing dim for broadcasting.
        x = dt.float().unsqueeze(-1) * self.basis_freq + self.phase
        return torch.cos(x)

In [ ]:
class EventFeatureEncoder(nn.Module):
    # Build a single event vector from cat ids and numeric features.
    # Input shapes (for a batch of B events):
    #   cat_ids: dict {field_name: LongTensor[B]}
    #   num:     FloatTensor[B, F_num]
    # Output shape: FloatTensor[B, out_dim]
    def __init__(self, cat_vocab: Dict[str, int], cat_emb_dim: int,
                 num_numeric: int, hidden: int, out_dim: int):
        super().__init__()
        self.cat_names = list(cat_vocab.keys())
        self.embeddings = nn.ModuleDict({
            n: nn.Embedding(num_embeddings=v, embedding_dim=cat_emb_dim)
            for n, v in cat_vocab.items()
        })
        in_dim = cat_emb_dim * len(self.cat_names) + num_numeric
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, cat_ids: Dict[str, torch.Tensor], num: torch.Tensor) -> torch.Tensor:
        embs = [self.embeddings[n](cat_ids[n]) for n in self.cat_names]
        x = torch.cat(embs + [num], dim=-1)
        return self.mlp(x)

In [ ]:
class HostMemoryModule(nn.Module):
    """Per-event message + GRU update of the host memory table.

    `mem_table` lives on CPU — it's mutable state, not a big matmul operand, so
    parking it on GPU mostly wasted VRAM. Per event we move two endpoint rows
    to the device, run message + GRU, write detached rows back; history keeps
    CPU snapshots of counterpart memory before each update.
    """

    def __init__(self, mem_dim: int, event_dim: int, time_dim: int):
        super().__init__()
        self.mem_dim = mem_dim
        self.time_enc = TimeEncoding(time_dim)
        msg_in = mem_dim * 2 + event_dim + time_dim
        self.msg_mlp = nn.Sequential(
            nn.Linear(msg_in, mem_dim),
            nn.ReLU(),
            nn.Linear(mem_dim, mem_dim),
        )
        self.gru = nn.GRUCell(input_size=mem_dim, hidden_size=mem_dim)

    def message(self, self_mem: torch.Tensor, other_mem: torch.Tensor,
                event_feat: torch.Tensor, dt_self: torch.Tensor) -> torch.Tensor:
        # All shapes: [B, *]. Inputs are expected to live on the same device
        # (typically the GPU; this is called from the per-event update below).
        te = self.time_enc(dt_self)
        return self.msg_mlp(torch.cat([self_mem, other_mem, event_feat, te], dim=-1))

    def update_one(self,
                   mem_table_cpu: torch.Tensor,
                   src_id: int, dst_id: int,
                   event_feat_dev: torch.Tensor,
                   dt_src_dev: torch.Tensor, dt_dst_dev: torch.Tensor,
                   device: torch.device,
                   ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Apply one event's update to the CPU memory table.

        Inputs:
          mem_table_cpu: [N_HOSTS, mem_dim] CPU float tensor (mutated in place)
          src_id, dst_id: int host ids
          event_feat_dev: [1, event_dim] on `device`
          dt_src_dev, dt_dst_dev: [1] on `device`, time gaps for the message
          device: the device on which the per-event neural computation runs

        Returns:
          mem_table_cpu (same object, with two rows updated)
          src_pre_cpu:  [mem_dim] CPU snapshot of src memory BEFORE the update
          dst_pre_cpu:  [mem_dim] CPU snapshot of dst memory BEFORE the update

        We deliberately move only two `mem_dim`-sized vectors per event to
        the GPU (instead of indexing into a GPU-resident table). On the way
        back we detach + .cpu() before writing into `mem_table_cpu`, so the
        table remains a pure CPU data buffer with no autograd graph.
        """
        # Read CPU rows for the two endpoints (cheap: 2 * mem_dim floats).
        src_pre_cpu = mem_table_cpu[src_id].clone().unsqueeze(0)   # [1, mem_dim] CPU
        dst_pre_cpu = mem_table_cpu[dst_id].clone().unsqueeze(0)   # [1, mem_dim] CPU

        # Move ONLY these two small vectors to the device for the GRU/message.
        src_pre = src_pre_cpu.to(device, non_blocking=True)
        dst_pre = dst_pre_cpu.to(device, non_blocking=True)

        msg_s = self.message(src_pre, dst_pre, event_feat_dev, dt_src_dev)
        msg_d = self.message(dst_pre, src_pre, event_feat_dev, dt_dst_dev)
        new_src = self.gru(msg_s, src_pre)                         # [1, mem_dim] dev
        new_dst = self.gru(msg_d, dst_pre)                         # [1, mem_dim] dev

        # Write back to the CPU table as detached data (no autograd link).
        mem_table_cpu[src_id] = new_src.detach().squeeze(0).cpu()
        mem_table_cpu[dst_id] = new_dst.detach().squeeze(0).cpu()

        # Snapshots returned for the history cache are CPU-resident data.
        return mem_table_cpu, src_pre_cpu.squeeze(0), dst_pre_cpu.squeeze(0)

## 10. Temporal attention

At scoring time for host \(i\), I pull its last \(K\) history entries (within the history span). Each slot carries the counterpart's **stored** memory at event time, the **current** encoder output for that event (so gradients reach the encoder), and a time delta encoding.

Attention uses current memory as query; masked where history is short. The attended vector gets concatenated with live memory and passed through a combine MLP before the risk head.


In [ ]:
class TemporalAttention(nn.Module):
    # Single-head scaled dot-product attention over K context vectors.
    # Inputs:
    #   query_mem:        [B, mem_dim]     (host memory used as query)
    #   ctx_counterpart:  [B, K, mem_dim]  (counterpart memory snapshots)
    #   ctx_event_feat:   [B, K, event_dim]
    #   ctx_dt:           [B, K]
    #   mask:             [B, K] bool (True = valid slot)
    # Output:
    #   [B, out_dim]
    def __init__(self, mem_dim: int, event_dim: int, time_dim: int,
                 hidden: int, out_dim: int):
        super().__init__()
        self.time_enc = TimeEncoding(time_dim)
        self.ctx_dim = mem_dim + event_dim + time_dim
        self.q_proj = nn.Linear(mem_dim, hidden)
        self.k_proj = nn.Linear(self.ctx_dim, hidden)
        self.v_proj = nn.Linear(self.ctx_dim, hidden)
        self.out_proj = nn.Linear(hidden, out_dim)
        self.scale = hidden ** 0.5

    def forward(self, query_mem: torch.Tensor,
                ctx_counterpart: torch.Tensor,
                ctx_event_feat: torch.Tensor,
                ctx_dt: torch.Tensor,
                mask: torch.Tensor) -> torch.Tensor:
        te = self.time_enc(ctx_dt)                                  # [B, K, time_dim]
        ctx = torch.cat([ctx_counterpart, ctx_event_feat, te], dim=-1)  # [B, K, ctx_dim]
        Q = self.q_proj(query_mem).unsqueeze(1)                     # [B, 1, H]
        Kp = self.k_proj(ctx)                                       # [B, K, H]
        Vp = self.v_proj(ctx)                                       # [B, K, H]
        scores = (Q * Kp).sum(-1) / self.scale                      # [B, K]

        # Mask invalid slots. If ALL slots are invalid, set scores to 0
        # before softmax to avoid NaNs and zero out the output below.
        all_invalid = (~mask).all(dim=-1, keepdim=True)             # [B, 1]
        scores = scores.masked_fill(~mask, float("-inf"))
        scores = torch.where(all_invalid.expand_as(scores),
                             torch.zeros_like(scores), scores)
        attn = F.softmax(scores, dim=-1)                            # [B, K]
        attn = torch.where(all_invalid.expand_as(attn),
                           torch.zeros_like(attn), attn)
        out = (attn.unsqueeze(-1) * Vp).sum(dim=1)                  # [B, H]
        return self.out_proj(out)                                   # [B, out_dim]

## 11. Risk head

Small MLP to one logit. Training uses `BCEWithLogitsLoss`; for inspection I sigmoid outside.


In [ ]:
class RiskHead(nn.Module):
    def __init__(self, in_dim: int, hidden: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.net(h).squeeze(-1)

In [ ]:
class TemporalHostRiskModel(nn.Module):
    def __init__(self, cat_vocab: Dict[str, int], num_numeric: int, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.event_encoder = EventFeatureEncoder(
            cat_vocab=cat_vocab,
            cat_emb_dim=cfg.cat_emb_dim,
            num_numeric=num_numeric,
            hidden=cfg.event_repr_dim,
            out_dim=cfg.event_repr_dim,
        )
        self.memory_module = HostMemoryModule(
            mem_dim=cfg.memory_dim,
            event_dim=cfg.event_repr_dim,
            time_dim=cfg.time_enc_dim,
        )
        self.attention = TemporalAttention(
            mem_dim=cfg.memory_dim,
            event_dim=cfg.event_repr_dim,
            time_dim=cfg.time_enc_dim,
            hidden=cfg.attn_hidden_dim,
            out_dim=cfg.host_emb_dim,
        )
        self.combine = nn.Sequential(
            nn.Linear(cfg.memory_dim + cfg.host_emb_dim, cfg.host_emb_dim),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
        )
        self.risk_head = RiskHead(cfg.host_emb_dim, cfg.risk_hidden_dim, cfg.dropout)

    # --- Convenience wrappers; the training loop calls these directly ---
    def encode_events(self, cat_ids: Dict[str, torch.Tensor],
                      num: torch.Tensor) -> torch.Tensor:
        return self.event_encoder(cat_ids, num)

    def predict(self, query_mem: torch.Tensor,
                ctx_counterpart: torch.Tensor,
                ctx_event_feat: torch.Tensor,
                ctx_dt: torch.Tensor,
                mask: torch.Tensor) -> torch.Tensor:
        c = self.attention(query_mem, ctx_counterpart, ctx_event_feat, ctx_dt, mask)
        h = self.combine(torch.cat([query_mem, c], dim=-1))
        return self.risk_head(h)

## 12. Training examples (conceptually)

Phase 1 already turned logs into tensors + indices. Phase 2 is **replay**: march time forward, update memory/history, and whenever the clock hits a decision time, score cached active hosts.

That pattern avoids building the full host×time grid in RAM — scoring rides along the simulated clock.


In [ ]:
# host_history[h] is a deque of dicts:
#   {"event_idx": int, "t_event": int, "counterpart_mem": Tensor[mem_dim] CPU}
# All snapshots in the history cache live on CPU. We only move the
# assembled context batch to the device once, right before attention.
host_history: List[deque] = [deque(maxlen=CFG.K_recent_events) for _ in range(N_HOSTS)]


def reset_temporal_state(model: TemporalHostRiskModel, cfg: Config
                         ) -> Tuple[torch.Tensor, List[deque]]:
    """Fresh memory and empty history (call at the start of each epoch).

    Returns a CPU float table — same rationale as `HostMemoryModule`: it's a
    state buffer, not layer weights, so we leave VRAM for the actual model.
    """
    mem_table_cpu = torch.zeros(N_HOSTS, cfg.memory_dim)
    history = [deque(maxlen=cfg.K_recent_events) for _ in range(N_HOSTS)]
    return mem_table_cpu, history


def build_context_batch(model: TemporalHostRiskModel,
                        mem_table_cpu: torch.Tensor,
                        history: List[deque],
                        hosts_cpu: torch.Tensor,
                        dt: int,
                        cfg: Config,
                        device: torch.device,
                        ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor,
                                   torch.Tensor, torch.Tensor]:
    """Build attention inputs for a batch of hosts at decision time `dt`.

    Performance design:
    - `query_mem`, `ctx_counterpart`, `ctx_dt`, and `mask` are first
      assembled on CPU from the host history cache (which is CPU-resident).
      Each one is then moved to the device in **one** transfer instead of
      being filled element-by-element with many tiny GPU writes from the
      Python loop.
    - All referenced event indices are gathered into a single 1-D tensor,
      passed through the *cached* device-side event tensors
      (`event_cat_dev` / `event_num_dev` — no `.to(DEVICE)` on the full
      tensor every call), encoded ONCE with the current
      `EventFeatureEncoder` (so gradients still flow through the encoder
      for context events during training), and scattered back into a
      `[B, K, event_repr_dim]` buffer with a single advanced-indexing
      assignment.

    Inputs:
      hosts_cpu: [B] int64 host ids on CPU (CPU index for the CPU mem_table)
      dt:        decision time in seconds
      device:    the device on which the model lives

    Returns five tensors (all on `device`):
      query_mem       [B, mem_dim]
      ctx_counterpart [B, K, mem_dim]
      ctx_event_feat  [B, K, event_repr_dim]
      ctx_dt          [B, K]
      mask            [B, K]   bool
    """
    B = int(hosts_cpu.shape[0])
    K = cfg.K_recent_events
    mem_dim = cfg.memory_dim

    # ---- 1) CPU staging of the per-host context buffers ------------------
    query_mem_cpu = mem_table_cpu.index_select(0, hosts_cpu)        # [B, mem_dim]
    ctx_counterpart_cpu = torch.zeros(B, K, mem_dim)
    ctx_dt_cpu = torch.zeros(B, K)
    mask_cpu = torch.zeros(B, K, dtype=torch.bool)

    flat_event_idxs: List[int] = []
    flat_b: List[int] = []
    flat_k: List[int] = []

    hosts_list = hosts_cpu.tolist()
    for b in range(B):
        h = hosts_list[b]
        recs = list(history[h])[-K:]
        for k_idx, rec in enumerate(recs):
            ctx_counterpart_cpu[b, k_idx] = rec["counterpart_mem"]
            ctx_dt_cpu[b, k_idx] = float(dt - rec["t_event"])
            mask_cpu[b, k_idx] = True
            flat_event_idxs.append(rec["event_idx"])
            flat_b.append(b)
            flat_k.append(k_idx)

    # ---- 2) Single-shot transfers to device ------------------------------
    query_mem = query_mem_cpu.to(device, non_blocking=True)
    ctx_counterpart = ctx_counterpart_cpu.to(device, non_blocking=True)
    ctx_dt = ctx_dt_cpu.to(device, non_blocking=True)
    mask = mask_cpu.to(device, non_blocking=True)

    # ---- 3) Encode ALL referenced events in one batched pass --------------
    if flat_event_idxs:
        idx_dev = torch.as_tensor(flat_event_idxs, dtype=torch.long, device=device)
        cat_batch = {f: event_cat_dev[f].index_select(0, idx_dev) for f in CAT_FIELDS}
        num_batch = event_num_dev.index_select(0, idx_dev)
        feats = model.encode_events(cat_batch, num_batch)            # [N, event_dim]
        ctx_event_feat = torch.zeros(B, K, feats.size(-1), device=device)
        b_t = torch.as_tensor(flat_b, dtype=torch.long, device=device)
        k_t = torch.as_tensor(flat_k, dtype=torch.long, device=device)
        ctx_event_feat[b_t, k_t] = feats
    else:
        ctx_event_feat = torch.zeros(B, K, cfg.event_repr_dim, device=device)

    return query_mem, ctx_counterpart, ctx_event_feat, ctx_dt, mask

## 13. Training loop

Rough picture:

- rewind memory/history each epoch;
- for each decision time in order: replay events since the previous decision (encoded in chunks for GPU memory), update CPU-side memory table, then score hosts pulled from the precomputed cache;
- accumulate loss over `decisions_per_grad_step` TBPTT-style steps and step the optimizer.

Gradients mainly flow through **context event encoding** + attention + head — the replay GRU writes detached CPU state so the graph doesn't explode across the whole stream.

**Class imbalance:** positives are rare; `pos_weight` comes straight from cached train labels.

See code comments in `train_one_epoch` for the exact tensor choreography.


In [ ]:
def chronological_event_indices(t_arr_np: np.ndarray, t_lo: int, t_hi: int) -> np.ndarray:
    """Indices of events with t_lo < t <= t_hi (the (prev_dt, dt] window)."""
    i_lo = np.searchsorted(t_arr_np, t_lo, side="right")
    i_hi = np.searchsorted(t_arr_np, t_hi, side="right")
    return np.arange(i_lo, i_hi)


def replay_through_time(model: TemporalHostRiskModel,
                        mem_table_cpu: torch.Tensor,
                        history: List[deque],
                        from_t: int, to_t: int,
                        cfg: Config,
                        update_with_grad: bool,
                        device: torch.device,
                        ) -> torch.Tensor:
    """Process events with `from_t < t <= to_t`, updating CPU memory + history.

    Encode in chunks (`cfg.replay_encode_chunk_size`) so long windows don't
    spike GPU memory; GRU updates stay sequential when the same host appears
    back-to-back. Read event features from `event_cat_dev` / `event_num_dev`
    (no full-matrix upload each call). Stage time gaps in NumPy for the chunk,
    then slice on device inside the per-event loop.
    """
    idxs = chronological_event_indices(t_arr, from_t, to_t)
    n_total = len(idxs)
    if n_total == 0:
        return mem_table_cpu

    # CPU bookkeeping for time gaps. We use a single int64 vector to track
    # last-seen timestamps across chunks within this replay call.
    SENT = float(cfg.history_window_sec * 4)
    last_seen_local = np.full(N_HOSTS, -1, dtype=np.int64)

    chunk_size = max(1, int(cfg.replay_encode_chunk_size))

    for chunk_start in range(0, n_total, chunk_size):
        chunk_end = min(chunk_start + chunk_size, n_total)
        chunk_idxs = idxs[chunk_start:chunk_end]
        chunk_idxs_t_cpu = torch.from_numpy(chunk_idxs).long()
        chunk_idxs_t_dev = chunk_idxs_t_cpu.to(device, non_blocking=True)

        # ---- 1) Encode the entire chunk in ONE pass on the device --------
        cat_batch = {f: event_cat_dev[f].index_select(0, chunk_idxs_t_dev)
                     for f in CAT_FIELDS}
        num_batch = event_num_dev.index_select(0, chunk_idxs_t_dev)
        if update_with_grad:
            ev_feats = model.encode_events(cat_batch, num_batch)   # [chunk, evt_dim]
        else:
            with torch.no_grad():
                ev_feats = model.encode_events(cat_batch, num_batch)

        # ---- 2) Pre-compute per-event dt arrays on CPU, then upload ------
        src_local = event_src.index_select(0, chunk_idxs_t_cpu).numpy()
        dst_local = event_dst.index_select(0, chunk_idxs_t_cpu).numpy()
        t_local = event_t.index_select(0, chunk_idxs_t_cpu).numpy()
        n_chunk = len(chunk_idxs)

        dt_s_arr = np.empty(n_chunk, dtype=np.float32)
        dt_d_arr = np.empty(n_chunk, dtype=np.float32)
        for j in range(n_chunk):
            s = int(src_local[j]); d = int(dst_local[j]); t = int(t_local[j])
            prev_s = last_seen_local[s]
            prev_d = last_seen_local[d]
            dt_s_arr[j] = SENT if prev_s < 0 else max(0, t - prev_s)
            dt_d_arr[j] = SENT if prev_d < 0 else max(0, t - prev_d)
            last_seen_local[s] = t
            last_seen_local[d] = t
        dt_s_dev = torch.from_numpy(dt_s_arr).to(device, non_blocking=True)
        dt_d_dev = torch.from_numpy(dt_d_arr).to(device, non_blocking=True)

        # ---- 3) Apply the per-event GRU update sequentially --------------
        # `update_one` reads two CPU rows, moves them to GPU, runs the
        # message + GRU, and writes detached results back to the CPU table.
        for j in range(n_chunk):
            e_idx = int(chunk_idxs[j])
            s = int(src_local[j]); d = int(dst_local[j]); t = int(t_local[j])
            ev_feat = ev_feats[j:j + 1]
            dt_s = dt_s_dev[j:j + 1]
            dt_d = dt_d_dev[j:j + 1]

            if update_with_grad:
                mem_table_cpu, src_pre_cpu, dst_pre_cpu = (
                    model.memory_module.update_one(
                        mem_table_cpu, s, d, ev_feat, dt_s, dt_d, device
                    )
                )
            else:
                with torch.no_grad():
                    mem_table_cpu, src_pre_cpu, dst_pre_cpu = (
                        model.memory_module.update_one(
                            mem_table_cpu, s, d, ev_feat, dt_s, dt_d, device
                        )
                    )

            # History entries: COUNTERPART pre-event memory as CPU data.
            history[s].append({"event_idx": e_idx, "t_event": t,
                               "counterpart_mem": dst_pre_cpu})
            history[d].append({"event_idx": e_idx, "t_event": t,
                               "counterpart_mem": src_pre_cpu})

    return mem_table_cpu

In [ ]:
from tqdm.auto import tqdm


def compute_pos_weight_from_split(
    examples: Dict[int, Tuple[np.ndarray, np.ndarray]],
) -> float:
    """Class-imbalance `pos_weight` from cached labels (sum over examples).

    Previously this rescanned hosts per decision time; now it's a cheap numpy
    reduction on the precomputed cache.
    """
    pos = 0
    total = 0
    for _, labels in examples.values():
        pos += int(labels.sum())
        total += int(labels.size)
    neg = total - pos
    pos = max(pos, 1)
    return float(neg) / float(pos)


def train_one_epoch(model: TemporalHostRiskModel,
                    optimizer: torch.optim.Optimizer,
                    loss_fn: nn.Module,
                    mem_table_cpu: torch.Tensor,
                    history: List[deque],
                    decision_times_train: np.ndarray,
                    examples: Dict[int, Tuple[np.ndarray, np.ndarray]],
                    cfg: Config,
                    epoch: int,
                    device: torch.device,
                    ) -> Tuple[torch.Tensor, List[deque], float]:
    """One training epoch over the chronological train period.

    Hosts and labels come from the precomputed `examples` map. The CPU memory
    table isn't part of the autograd graph, so no `.detach()` gymnastics.
    """
    model.train()
    losses_in_epoch: List[float] = []
    accum_loss = torch.zeros((), device=device)
    n_accum = 0
    optimizer.zero_grad()
    # Start at T_MIN - 1 so events with t == T_MIN are included in the first
    # replay window. (chronological_event_indices uses an exclusive lower bound.)
    prev_t = T_MIN - 1

    pbar = tqdm(
        enumerate(decision_times_train),
        total=len(decision_times_train),
        desc=f"Epoch {epoch}/{cfg.num_epochs} (decision times)",
        unit="dt",
        leave=False,
        dynamic_ncols=True,
    )
    for _, dt in pbar:
        dt = int(dt)
        # 1) Replay events (prev_t, dt]. We pass `update_with_grad=False`
        #    because the new CPU `mem_table` is written via `.detach()`,
        #    so there is no autograd path from the GRU/msg through `mem_table`
        #    back to the loss. Gradients still flow into the event encoder
        #    via `build_context_batch` (which re-encodes the context events
        #    with grad enabled), and into the attention + combine + risk
        #    head via the prediction. Skipping the replay-time graph saves
        #    a large amount of GPU memory on long replay windows.
        mem_table_cpu = replay_through_time(
            model, mem_table_cpu, history, prev_t, dt, cfg,
            update_with_grad=False, device=device,
        )
        # 2) Score active hosts at dt — read from precomputed cache
        ex = examples.get(dt)
        if ex is not None:
            hosts_np, labels_np = ex
            hosts_cpu = torch.from_numpy(hosts_np.astype(np.int64))
            label_t = torch.from_numpy(labels_np.astype(np.float32)).to(device)

            qm, cc, cf, cd, mk = build_context_batch(
                model, mem_table_cpu, history, hosts_cpu, dt, cfg, device,
            )
            logits = model.predict(qm, cc, cf, cd, mk)
            loss = loss_fn(logits, label_t)
            accum_loss = accum_loss + loss
            n_accum += 1

        # 3) Periodic backward; mem_table is CPU data so no detach is needed.
        if n_accum >= cfg.decisions_per_grad_step:
            (accum_loss / n_accum).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            step_loss = float(accum_loss.detach().item()) / n_accum
            losses_in_epoch.append(step_loss)
            pbar.set_postfix(loss=f"{step_loss:.4f}", grad_steps=len(losses_in_epoch))
            optimizer.zero_grad()
            accum_loss = torch.zeros((), device=device)
            n_accum = 0

        prev_t = dt

    if n_accum > 0:
        (accum_loss / n_accum).backward()
        optimizer.step()
        step_loss = float(accum_loss.detach().item()) / n_accum
        losses_in_epoch.append(step_loss)
        pbar.set_postfix(loss=f"{step_loss:.4f}", grad_steps=len(losses_in_epoch))

    avg = float(np.mean(losses_in_epoch)) if losses_in_epoch else float("nan")
    if cfg.verbose:
        print(f"[epoch {epoch}] train loss avg = {avg:.4f} over "
              f"{len(losses_in_epoch)} grad steps")
    return mem_table_cpu, history, avg

## 14. Validation + early stopping

After training chunks of decision times, I roll forward through validation **without** optimiser gradients, still advancing memory so test starts from an honest post-val state. Early stopping watches validation AUROC (or whatever metric you wire in).

No shuffling timeline chunks; that would leak future into past.


In [ ]:
from tqdm.auto import tqdm


@torch.no_grad()
def evaluate_split(model: TemporalHostRiskModel,
                   mem_table_cpu: torch.Tensor,
                   history: List[deque],
                   decision_times_eval: np.ndarray,
                   examples: Dict[int, Tuple[np.ndarray, np.ndarray]],
                   cfg: Config,
                   prev_t_init: int,
                   device: torch.device,
                   loss_fn: Optional[nn.Module] = None,
                   ) -> Tuple[torch.Tensor, List[deque], Dict[str, float],
                              np.ndarray, np.ndarray]:
    """Evaluate on a chronological split, replaying memory state without grad.

    Same temporal discipline as before — still chaining from `prev_t_init` —
    but hosts/labels come from the `examples` cache. Replay uses the device
    event tensors (no full-matrix `.to(device)` per step).
    """
    model.eval()
    all_probs: List[float] = []
    all_labels: List[int] = []
    losses: List[float] = []
    prev_t = int(prev_t_init)

    eval_bar = tqdm(
        decision_times_eval,
        total=len(decision_times_eval),
        desc="Eval (decision times)",
        unit="dt",
        leave=False,
        dynamic_ncols=True,
    )
    for dt in eval_bar:
        dt = int(dt)
        mem_table_cpu = replay_through_time(
            model, mem_table_cpu, history, prev_t, dt, cfg,
            update_with_grad=False, device=device,
        )
        ex = examples.get(dt)
        if ex is None:
            prev_t = dt
            continue
        hosts_np, labels_np = ex
        hosts_cpu = torch.from_numpy(hosts_np.astype(np.int64))
        label_t = torch.from_numpy(labels_np.astype(np.float32)).to(device)

        qm, cc, cf, cd, mk = build_context_batch(
            model, mem_table_cpu, history, hosts_cpu, dt, cfg, device,
        )
        logits = model.predict(qm, cc, cf, cd, mk)
        if loss_fn is not None:
            batch_loss = float(loss_fn(logits, label_t).item())
            losses.append(batch_loss)
            eval_bar.set_postfix(loss=f"{batch_loss:.4f}")
        all_probs.extend(torch.sigmoid(logits).detach().cpu().tolist())
        all_labels.extend(int(x) for x in labels_np.tolist())
        prev_t = dt

    probs = np.asarray(all_probs, dtype=np.float64)
    labels_arr = np.asarray(all_labels, dtype=np.int64)
    metrics: Dict[str, float] = {
        "loss": float(np.mean(losses)) if losses else float("nan")
    }
    if labels_arr.size > 0 and 0 < labels_arr.sum() < labels_arr.size:
        metrics["auroc"] = float(roc_auc_score(labels_arr, probs))
        metrics["ap"] = float(average_precision_score(labels_arr, probs))
    else:
        metrics["auroc"] = float("nan")
        metrics["ap"] = float("nan")
    return mem_table_cpu, history, metrics, probs, labels_arr

### Sanity check before the long run

Synthetic batch through the model to catch shape bugs before paying for full replay.


In [ ]:
def build_model(cfg: Config) -> TemporalHostRiskModel:
    model = TemporalHostRiskModel(cat_vocab_sizes, num_numeric=event_num.size(-1), cfg=cfg).to(DEVICE)
    return model


# --- Sanity pass on a tiny synthetic batch ---
sanity_model = build_model(CFG)
sanity_model.eval()
with torch.no_grad():
    B = 4
    qm = torch.randn(B, CFG.memory_dim, device=DEVICE)
    cc = torch.randn(B, CFG.K_recent_events, CFG.memory_dim, device=DEVICE)
    cf = torch.randn(B, CFG.K_recent_events, CFG.event_repr_dim, device=DEVICE)
    cd = torch.rand(B, CFG.K_recent_events, device=DEVICE) * 3600.0
    mk = torch.ones(B, CFG.K_recent_events, dtype=torch.bool, device=DEVICE)
    logits = sanity_model.predict(qm, cc, cf, cd, mk)
    print("Sanity logits shape:", logits.shape, "(expected [B])")
    print("Sanity logits sample:", logits[:4].cpu().numpy())

# Encoder sanity
sanity_cat = {f: torch.zeros(2, dtype=torch.long, device=DEVICE) for f in CAT_FIELDS}
sanity_num = torch.zeros(2, event_num.size(-1), device=DEVICE)
sanity_evt = sanity_model.encode_events(sanity_cat, sanity_num)
print("Encoded event shape:", sanity_evt.shape, "(expected [2, event_repr_dim])")

### Full training

Outer loop over epochs: reset state, train segment, validate segment, checkpoint when val improves, stop after patience runs out.


In [ ]:
from tqdm.auto import tqdm

# DEC_TRAIN / DEC_VAL / DEC_TEST and the per-split example dictionaries
# (train_examples, val_examples, test_examples) were precomputed in the
# "decision-example cache" cell above. We use them directly here.

print("Computing pos_weight from cached training labels (instant)...")
POS_WEIGHT = compute_pos_weight_from_split(train_examples)
print(f"pos_weight = {POS_WEIGHT:.2f}")

model = build_model(CFG)
optimizer = torch.optim.Adam(model.parameters(),
                             lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEVICE))

best_state = None
best_val_auroc = -float("inf")
patience_left = CFG.early_stop_patience
history_train_loss: List[float] = []
history_val_metric: List[Dict[str, float]] = []

epoch_loop = tqdm(range(1, CFG.num_epochs + 1), desc="Epochs", dynamic_ncols=True)
for epoch in epoch_loop:
    mem_table, history = reset_temporal_state(model, CFG)

    # Train across train period (consumes the cached train_examples)
    mem_table, history, train_loss = train_one_epoch(
        model, optimizer, loss_fn, mem_table, history,
        DEC_TRAIN, train_examples, CFG, epoch, DEVICE,
    )

    # Validation continues from the state after training, using val_examples
    mem_table, history, val_metrics, val_probs, val_labels = evaluate_split(
        model, mem_table, history, DEC_VAL, val_examples, CFG,
        prev_t_init=int(DEC_TRAIN[-1]) if len(DEC_TRAIN) > 0 else T_MIN - 1,
        device=DEVICE, loss_fn=loss_fn,
    )

    val_auroc = val_metrics.get("auroc", float("nan"))
    epoch_loop.set_postfix(
        train_loss=f"{train_loss:.4f}",
        val_auroc=f"{val_auroc:.4f}" if not math.isnan(val_auroc) else "nan",
        refresh=False,
    )

    history_train_loss.append(train_loss)
    history_val_metric.append(val_metrics)
    print(f"[epoch {epoch}] val: "
          + ", ".join(f"{k}={v:.4f}" for k, v in val_metrics.items()))

    if (not math.isnan(val_metrics.get("auroc", float("nan")))
            and val_metrics["auroc"] > best_val_auroc):
        best_val_auroc = val_metrics["auroc"]
        best_state = {k: v.detach().cpu().clone()
                      for k, v in model.state_dict().items()}
        patience_left = CFG.early_stop_patience
        print(f"  -> new best val AUROC = {best_val_auroc:.4f}, model saved")
    else:
        patience_left -= 1
        print(f"  -> no improvement; patience_left = {patience_left}")
        if patience_left <= 0:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)
print(f"Best val AUROC: {best_val_auroc:.4f}")

## 15. Metrics + plots

Replay val then test so memory lines up, then standard classification curves / confusion / calibration-ish histograms plus loss traces.

Plots read cached splits where relevant so I'm not recomputing labels in plotting code.


In [ ]:
# Replay through train+val (no grad) to land at the start of the test period.
# All three calls read from the corresponding precomputed example caches —
# we never recompute active hosts or labels here.
mem_table, history = reset_temporal_state(model, CFG)
mem_table, history, _, _, _ = evaluate_split(
    model, mem_table, history, DEC_TRAIN, train_examples, CFG,
    prev_t_init=T_MIN - 1, device=DEVICE, loss_fn=None,
)
mem_table, history, _, _, _ = evaluate_split(
    model, mem_table, history, DEC_VAL, val_examples, CFG,
    prev_t_init=int(DEC_TRAIN[-1]) if len(DEC_TRAIN) > 0 else T_MIN - 1,
    device=DEVICE, loss_fn=None,
)
mem_table, history, test_metrics, test_probs, test_labels = evaluate_split(
    model, mem_table, history, DEC_TEST, test_examples, CFG,
    prev_t_init=int(DEC_VAL[-1]) if len(DEC_VAL) > 0 else T_MIN - 1,
    device=DEVICE, loss_fn=loss_fn,
)
print("Test metrics:", test_metrics)


THRESHOLD = 0.5
y_pred_thresh = (test_probs >= THRESHOLD).astype(int)
if test_labels.sum() > 0:
    p, r, f1, _ = precision_recall_fscore_support(
        test_labels, y_pred_thresh, average="binary", zero_division=0
    )
    cm = confusion_matrix(test_labels, y_pred_thresh)
else:
    p = r = f1 = float("nan")
    cm = np.zeros((2, 2), dtype=int)
print(f"At threshold {THRESHOLD}: precision={p:.4f}, recall={r:.4f}, f1={f1:.4f}")
print("Confusion matrix [[TN, FP], [FN, TP]]:\n", cm)

In [ ]:
# --- Plots --------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# (a) training / validation curves
ax = axes[0, 0]
ax.plot(range(1, len(history_train_loss) + 1), history_train_loss,
        marker="o", label="train loss")
val_aurocs = [m.get("auroc", float("nan")) for m in history_val_metric]
val_losses = [m.get("loss", float("nan")) for m in history_val_metric]
ax.plot(range(1, len(val_losses) + 1), val_losses, marker="s", label="val loss")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.set_title("Loss curves")

# (b) val auroc curve
ax = axes[0, 1]
ax.plot(range(1, len(val_aurocs) + 1), val_aurocs, marker="o", color="darkgreen")
ax.set_ylim(0, 1); ax.set_xlabel("epoch"); ax.set_ylabel("val AUROC")
ax.set_title("Validation AUROC")

# (c) histogram of predicted probabilities (test)
ax = axes[0, 2]
if len(test_probs) > 0:
    ax.hist(test_probs[test_labels == 0], bins=30, alpha=0.6, label="negatives")
    if (test_labels == 1).any():
        ax.hist(test_probs[test_labels == 1], bins=30, alpha=0.8, label="positives", color="crimson")
ax.set_xlabel("predicted prob"); ax.set_ylabel("count"); ax.set_title("Test probabilities")
ax.legend()

# (d) ROC curve
ax = axes[1, 0]
if test_labels.sum() > 0 and test_labels.sum() < len(test_labels):
    fpr, tpr, _ = roc_curve(test_labels, test_probs)
    ax.plot(fpr, tpr)
    ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title(f"ROC (AUROC={test_metrics.get('auroc', float('nan')):.3f})")

# (e) PR curve
ax = axes[1, 1]
if test_labels.sum() > 0:
    pr, rc, _ = precision_recall_curve(test_labels, test_probs)
    ax.plot(rc, pr)
ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_title(f"Precision-Recall (AP={test_metrics.get('ap', float('nan')):.3f})")

# (f) class imbalance over decision times in TEST — reads from cache only
ax = axes[1, 2]
n_pos_per_dt: List[int] = []
n_hosts_per_dt: List[int] = []
for dt in DEC_TEST:
    ex = test_examples.get(int(dt))
    if ex is None:
        n_pos_per_dt.append(0); n_hosts_per_dt.append(0); continue
    hosts_np, labels_np = ex
    n_pos_per_dt.append(int(labels_np.sum()))
    n_hosts_per_dt.append(int(hosts_np.size))
ax.plot(DEC_TEST, n_hosts_per_dt, label="active hosts", color="steelblue")
ax.plot(DEC_TEST, n_pos_per_dt, label="positives", color="crimson")
ax.set_xlabel("decision time"); ax.set_ylabel("count")
ax.set_title("Test: scored hosts vs positives")
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(CFG.figures_dir, "training_eval_panel.png"), dpi=120)
plt.show()

## 16. Example predictions

Ranked table of high-risk host-times in test — the sort of thing you'd spot-check before trusting the model anywhere real.


In [ ]:
# Build a flat record (host, dt, prob, label) for the test split for inspection.
# This reads from the precomputed example caches (no per-host scanning).
records: List[Dict[str, float]] = []
mem_table, history = reset_temporal_state(model, CFG)
mem_table, history, _, _, _ = evaluate_split(
    model, mem_table, history, DEC_TRAIN, train_examples, CFG,
    prev_t_init=T_MIN - 1, device=DEVICE,
)
mem_table, history, _, _, _ = evaluate_split(
    model, mem_table, history, DEC_VAL, val_examples, CFG,
    prev_t_init=int(DEC_TRAIN[-1]) if len(DEC_TRAIN) > 0 else T_MIN - 1,
    device=DEVICE,
)

prev_t = int(DEC_VAL[-1]) if len(DEC_VAL) > 0 else T_MIN - 1
model.eval()
with torch.no_grad():
    for dt in DEC_TEST:
        dt = int(dt)
        mem_table = replay_through_time(
            model, mem_table, history, prev_t, dt, CFG,
            update_with_grad=False, device=DEVICE,
        )
        ex = test_examples.get(dt)
        if ex is None:
            prev_t = dt; continue
        hosts_np, labels_np = ex
        hosts_cpu = torch.from_numpy(hosts_np.astype(np.int64))
        qm, cc, cf, cd, mk = build_context_batch(
            model, mem_table, history, hosts_cpu, dt, CFG, DEVICE,
        )
        probs = torch.sigmoid(model.predict(qm, cc, cf, cd, mk)).cpu().numpy()
        for h, p, y in zip(hosts_np.tolist(), probs.tolist(), labels_np.tolist()):
            records.append({"dt": dt, "host_id": int(h), "host": id2host[int(h)],
                            "prob": float(p), "label": int(y)})
        prev_t = dt

results_df = pd.DataFrame(records).sort_values("prob", ascending=False)
print("Top-20 highest-risk predictions in test period:")
print(results_df.head(20))
print()
print("Top-10 confirmed-positive predictions ranked by prob:")
print(results_df[results_df["label"] == 1].head(10))

## Benchmarks

Optional timings: cache cold vs warm, one replay slice, one epoch on the demo settings — handy when you're wondering whether Python overhead or the actual forward pass dominates.


In [ ]:
# ---- Benchmark: per-stage wall-time so you can find the next bottleneck ----
import time
from contextlib import contextmanager


@contextmanager
def _timed(label: str):
    t0 = time.perf_counter()
    yield
    dt = time.perf_counter() - t0
    print(f"  {label:<55s} {dt:8.3f} s")


print("Benchmark (debug config):")

# (1) numeric features cache: warm load vs forced recompute
with _timed("(1a) load_or_compute_numeric_features (warm cache)"):
    _ = load_or_compute_numeric_features(events_df, CFG)

# Build a temporary CFG override so we don't permanently flip the user flag.
_cfg_force_feats = Config(**{**asdict(CFG), "force_recompute_features": True})
with _timed("(1b) compute_event_features_fast (cold, full recompute)"):
    _ = compute_event_features_fast(events_df, _cfg_force_feats)

# (2) decision-example cache: warm load vs cold recompute
with _timed("(2a) load_or_compute_decision_cache (warm cache)"):
    _ = load_or_compute_decision_cache(decision_times, CFG, n_events=len(t_arr))

with _timed("(2b) precompute_decision_examples (cold, in-memory)"):
    _ = precompute_decision_examples(decision_times, CFG)

# (3) one call to replay_through_time on a short interval
_bench_model = build_model(CFG)
_bench_mem, _bench_hist = reset_temporal_state(_bench_model, CFG)
_short_from = T_MIN - 1
_short_to = int(min(T_MAX, T_MIN + 30 * 60))  # first 30 minutes of events
with _timed(f"(3)  replay_through_time over [{_short_from}, {_short_to}]"):
    _bench_mem = replay_through_time(
        _bench_model, _bench_mem, _bench_hist,
        _short_from, _short_to, CFG, update_with_grad=False, device=DEVICE,
    )

# (4) one full training epoch on the debug config
_epoch_model = build_model(CFG)
_epoch_opt = torch.optim.Adam(_epoch_model.parameters(), lr=CFG.learning_rate,
                              weight_decay=CFG.weight_decay)
_epoch_pw = compute_pos_weight_from_split(train_examples)
_epoch_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(_epoch_pw, device=DEVICE)
)
_epoch_mem, _epoch_hist = reset_temporal_state(_epoch_model, CFG)
with _timed("(4)  train_one_epoch on debug config"):
    _epoch_mem, _epoch_hist, _ = train_one_epoch(
        _epoch_model, _epoch_opt, _epoch_loss_fn,
        _epoch_mem, _epoch_hist, DEC_TRAIN, train_examples, CFG,
        epoch=0, device=DEVICE,
    )

print("Benchmark complete.")

## Implementation notes

Design choices I’d revisit if I forked this:

**Hosts as nodes.** Lateral movement is mostly machine-to-machine; users ride inside events as attributes so we’re not exploding the graph with sparse user nodes.

**Event-driven memory.** Irregular timestamps matter; bucketizing into coarse bins throws away order inside the bucket. Per-event GRU bumps keep continuous-time-ish behavior.

**Periodic scoring.** Matches operational review cadence and keeps evaluation countable (finite host-time pairs per epoch).

**Attention over recent edges vs static k-hop.** Neighbors in a temporal graph are “who you recently talked to”, not an abstract adjacency matrix.

**Forecasting vs anomaly detection.** Labels live ahead of decision time on purpose.

**Counterpart snapshots.** Storing the other host's memory at interaction time is expensive if done naïvely; deques + detach’d snapshots are my pragmatic compromise.

**CPU `mem_table`.** Keeps GPU memory sane on large host vocabularies; downside is gradients don't train the replay GRU through memory writes — encoder + attention carry most of the signal in practice.

**Chunked replay encoding.** Batch-encode slices of events, still apply GRU strictly in order — avoids OOM on long windows.

**Cached decision examples.** Biggest iteration-time win: label/active-host work once per run.

**Device-resident event tensors.** Upload cat/num features once; index in-place instead of `.to(device)` every line.

**No PyG.** Could wrap later; clarity mattered more for this notebook.



## Limitations + next ideas

**Honest limits**

- Training on a subsample doesn't guarantee behavior on the full 58-day dump.
- Red-team alignment isn’t ground-truth compromise.
- History snapshots don't retroactively update when counterpart memory moves later — intentional temporality, but approximate.
- Extreme imbalance makes AUROC optimistic vs PR-AUC.
- No fancy negative sampling yet; might matter at enterprise scale.

**Things I'd try next**

- richer snapshots / compaction scheme
- heterogeneous graphs (user/process layers)
- smarter negatives + maybe self-supervised pretraining
- calibration (temperature scaling)
- multi-horizon heads

Config + artifacts paths keep runs reproducible enough for coursework / experiments.


In [ ]:
# Persist the trained model and the config for reproducibility.
torch.save({"state_dict": model.state_dict(),
            "cfg": asdict(CFG),
            "cat_vocab_sizes": cat_vocab_sizes,
            "num_numeric": event_num.size(-1)},
           os.path.join(CFG.artifacts_dir, "temporal_host_risk.pt"))
print("Saved model to", os.path.join(CFG.artifacts_dir, "temporal_host_risk.pt"))